In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [7]:
# Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True, transform=transform)
testset = CIFAR10(root="./data",train=False,download=True, transform=transform)

In [8]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [24]:
trainloader=DataLoader(trainset, batch_size=64, shuffle=True)
testloader= DataLoader(testset, batch_size=64)

# Build the CNN

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32 , kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64 , kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel size = 2, stride = 2

            nn.Conv2d(64, 128 , kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # kernel size = 2, stride = 2     
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x= self.fc_layers(x)
        return x
        
        

In [26]:
model = CNN()

In [27]:
criteria= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters())

## Training the CNN

In [28]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss= criteria(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    print(f"epoch = {epoch+1} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1 & loss = 1.374180192456526
epoch = 2 & loss = 0.9327903649843562
epoch = 3 & loss = 0.7501573558811032
epoch = 4 & loss = 0.6241321566769534
epoch = 5 & loss = 0.5150715055520577
epoch = 6 & loss = 0.42128278301728656
epoch = 7 & loss = 0.33613208654667714
epoch = 8 & loss = 0.25851590179688183
epoch = 9 & loss = 0.19495427472245358
epoch = 10 & loss = 0.15639598693107934


In [31]:
# Evaluate our CNN

correct_labels = 0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels*100}")

accuracy = 75.37
